# RobBERT mixed-language improvement sweep

This notebook keeps the supplied Dutch and English reviews in one model. It screens input/loss choices on one training fold, confirms the two best candidates with 5-fold × 3-seed cross-validation, and only then evaluates the selected configuration on the existing 960-row test set. Completed trials are resumable from Google Drive.

## 1. Mount Drive and upload `robbert_colab_source.zip`

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
uploaded = files.upload()
assert "robbert_colab_source.zip" in uploaded

## 2. Install the current project snapshot

In [ ]:
import shutil
from pathlib import Path

workspace = Path("/content/robbert_source")
if workspace.exists():
    shutil.rmtree(workspace)
shutil.unpack_archive("/content/robbert_colab_source.zip", workspace)
%cd /content/robbert_source
%pip install -q --ignore-requires-python -e '.[train,finetune]'

## 3. Verify GPU and frozen data contract

In [ ]:
import torch
import yaml

config_path = Path("configs/models/robbert_improvement.yaml")
config = yaml.safe_load(config_path.read_text())
assert torch.cuda.is_available(), "Select Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Language policy:", "single mixed Dutch + English model")
print("Candidates:", [row["id"] for row in config["screening"]["candidates"]])

## 4. Run or resume the staged sweep

The `trials/` cache is written after every fold/seed run. Re-running this cell resumes instead of repeating completed trials.

In [ ]:
output_dir = Path("/content/drive/MyDrive/Workspace365_assignment/robbert_improvement")
output_dir.mkdir(parents=True, exist_ok=True)
!sentiment-robbert-improve --config {config_path} --output-dir {output_dir}

## 5. Verify and display the decision evidence

In [ ]:
from dutch_sentiment.experiments.robbert_improvement import verify_bundle

result = verify_bundle(output_dir, config_path)
print("Promoted:", result["promoted_candidates"])
for row in result["confirmation"]:
    print(
        row["candidate_id"],
        "OOF Macro-F1:",
        round(row["oof_metrics"]["macro_f1"], 4),
        "CV std:",
        round(row["macro_f1_std"], 4),
    )
print("Winner:", result["winner"])
if result["final_test"]:
    print("Test metrics:", result["final_test"]["metrics"])

## 6. Local handoff

Retain the Drive directory, copy it to the repository if needed, then run `make robbert-improve-verify BUNDLE=...` and `make robbert-improve-import BUNDLE=...`. MLflow import remains local so Colab never edits the SQLite tracking database.